# Bayesian Modeling of Drum Pattern Features
In the preprocessing notebook, rhythmic and spectral features were extracted from the **Groove MIDI Dataset (GMD)** developed by Magenta.  
The original dataset contains over one thousand human-performed drum recordings with aligned MIDI and audio files.  
Each performance represents a short drum groove played by professional drummers across various musical styles.

Using the `librosa` Python library, several **audio features** were computed directly from the waveform of each recording, including:

- **Tempo (BPM)** and **Onset Rate** – rhythm speed and density  
- **Zero-Crossing Rate (ZCR)** and **RMS Energy** – percussiveness and loudness  
- **Spectral Centroid, Bandwidth, and Rolloff** – measures of brightness and tonal spread  
- **MFCCs (Mel-Frequency Cepstral Coefficients)** – 13 coefficients (means and standard deviations) summarizing timbre and spectral texture

After preprocessing, the final dataset (`gmd_audio_features.parquet`) contains one row per audio clip and one categorical label (`style`), representing the main drumming style of each performance.

---

## Modeling Overview

The goal of this notebook is to explore how Bayesian probabilistic models can classify drumming styles based on the extracted rhythmic and spectral features.

1. **Baseline Model – Naive Bayes:**  
   A simple Bayesian classifier that assumes feature independence.  
   This model serves as a reference point for evaluating more flexible Bayesian approaches.

2. **Extended Bayesian Models:**  
   After the baseline, additional Bayesian methods such as **Bayesian Multinomial Logistic Regression** will be implemented using `PyMC`.  
   These models will incorporate informative priors (based on real-world genre frequencies) and allow richer uncertainty estimation.

Throughout this notebook, model calibration, uncertainty quantification, and predictive accuracy will be compared to understand how well Bayesian inference captures the complex, overlapping nature of modern drumming styles.


In [3]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import arviz as az
import pymc as pm
import pymc.math as pm_math
import pytensor.tensor as pt
from scipy.stats import mode

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, top_k_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# Set random seeds for reproducibility
np.random.seed(42)

# Data Understanding & Preprocessing

In [4]:
# 1. Load Data
df = pd.DataFrame(pd.read_parquet("cleaned.parquet"))

# 2. Group Minority Classes
rare_styles = ["dance", "afrobeat", "blues", "middleeastern"]
df.loc[df["style"].isin(rare_styles), "style"] = "others"

# 3. Split Data
X = df.drop(columns=["audio_path", "style"])
y = df["style"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# 4. Scaling (Crucial for Bayesian Models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# 5. Label Encoding
classes = sorted(y.unique())
le = LabelEncoder().fit(classes)
y_train_int = le.transform(y_train)
y_val_int   = le.transform(y_val)
y_test_int  = le.transform(y_test)

K = len(classes)
print(f"Classes: {K} | Train Size: {X_train.shape} | Val Size: {X_val.shape}")

Classes: 14 | Train Size: (763, 33) | Val Size: (163, 33)


# Naive Bayes (Baseline)

In [ ]:
smoothing_values = [10**i for i in range(-5, 0)]
best_nb_acc = 0
best_smoothing = 0

for s in smoothing_values:
    nb = GaussianNB(var_smoothing=s)
    nb.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_val, nb.predict(X_val_scaled))
    if acc > best_nb_acc:
        best_nb_acc = acc
        best_smoothing = s

print(f"Baseline Naive Bayes Accuracy: {best_nb_acc:.2%} (smoothing={best_smoothing:.0e})")

Baseline Naive Bayes Accuracy: 21.47% (smoothing=1e-05)


The Gaussian Naive Bayes model reached about 21% validation accuracy, well above random guessing but limited by its independence and Gaussian assumptions.
Varying var_smoothing had no effect, and identical confusion matrices confirmed the model’s simplicity.
This baseline shows the features contain useful signal but also highlights the need for a more flexible Bayesian model.

# Baysian Models

## Define Priors
I would be comparing uniform, dataset genre portion, and realworld genre portion (Ref: https://www.musicbusinessworldwide.com/data/genre-split-of-total-on-demand-audio-streams-in-the-us-annual-via-luminate/?utm_source=chatgpt.com)

Baysian Model Priors

In [6]:
# 1. Uniform Prior (Flat)
prior_uniform = np.ones(K) / K
log_prior_uniform = np.log(prior_uniform)

# 2. Dataset Prior (Based on training counts)
counts = pd.Series(y_train_int).value_counts().sort_index()
counts = counts.reindex(range(K), fill_value=0) + 1  # Add 1 smoothing
prior_dataset = counts.values / counts.values.sum()
log_prior_dataset = np.log(prior_dataset)

# 3. Real-World Prior (External Knowledge)
real_world_map = {
    "r&b/hiphop": 0.253, "rock": 0.173, "pop": 0.122,
    "country": 0.087, "latin": 0.084, "others": 0.281
}

def get_realworld_prob(style_name):
    # Mapping logic
    style_name = style_name.lower()
    if style_name in ["hiphop", "soul", "funk", "gospel"]: return real_world_map["r&b/hiphop"] / 4
    if style_name in ["rock", "punk"]: return real_world_map["rock"] / 2
    if style_name in ["pop"]: return real_world_map["pop"]
    if style_name in ["country"]: return real_world_map["country"]
    if style_name in ["latin", "afrocuban"]: return real_world_map["latin"] / 2
    return real_world_map["others"] / 3 # jazz, etc.

prior_realworld = np.array([get_realworld_prob(c) for c in classes])
prior_realworld /= prior_realworld.sum()
log_prior_realworld = np.log(prior_realworld)

prior_experiments = {
    "Uniform": log_prior_uniform,
    "Dataset": log_prior_dataset,
    "Real-World": log_prior_realworld
}
print("Priors defined successfully.")

Priors defined successfully.


# PyMC Model (Linear Model Comparison)

In [ ]:
import pymc as pm
import pymc.math as pm_math
from sklearn.metrics import top_k_accuracy_score

grid_hidden = [5, 10]
grid_sigma  = [0.1, 0.5]
# Dictionary of Priors
prior_experiments = {
    "Uniform": log_prior_uniform,
    "Dataset": log_prior_dataset,
    "Real-World": log_prior_realworld
}

results_tuning = []
print(f"--- Starting Full Grid Search ---")

# Triple Loop
for name, prior_mu in prior_experiments.items():   # Loop over Priors
    for n_hid in grid_hidden:                      # Loop over Network Size
        for sig in grid_sigma:                     # Loop over Regularization

            coords = {
                "classes": classes,
                "features": [f"f{i}" for i in range(X_train_scaled.shape[1])],
                "hidden": [f"h{i}" for i in range(n_hid)]
            }

            with pm.Model(coords=coords) as tune_model:
                X_data = pm.Data("X_data", X_train_scaled)
                y_data = pm.Data("y_data", y_train_int)

                # Weights Prior (Regularization)
                w1 = pm.Normal("w1", 0, sig, dims=("features", "hidden"))
                b1 = pm.Normal("b1", 0, sig, dims="hidden")
                act1 = pm_math.tanh(pm_math.dot(X_data, w1) + b1)

                w2 = pm.Normal("w2", 0, sig, dims=("hidden", "classes"))

                # It now changes based on the loop (Uniform -> Dataset -> Real-World)
                b2 = pm.Normal("b2", mu=prior_mu, sigma=1.0, dims="classes")

                logits = pm_math.dot(act1, w2) + b2
                probs = pm.Deterministic("probs", pm_math.softmax(logits, axis=1))

                # Likelihood
                pm.Categorical("y_obs", logit_p=logits, observed=y_data)

                # Inference
                approx = pm.fit(n=30000, method='advi', progressbar=False, random_seed=42)
                trace = approx.sample(draws=500)

                # Evaluation
                pm.set_data({"X_data": X_val_scaled, "y_data": y_val_int})
                post_pred = pm.sample_posterior_predictive(trace, var_names=["probs"], progressbar=False)
                mean_probs = post_pred.posterior_predictive["probs"].mean(dim=["chain", "draw"]).values

                acc1 = top_k_accuracy_score(y_val_int, mean_probs, k=1)
                acc2 = top_k_accuracy_score(y_val_int, mean_probs, k=2)

                print(f"Prior: {name} | Hidden: {n_hid} | Sigma: {sig} | Top-1: {acc1:.2%} | Top-2: {acc2:.2%}")
                results_tuning.append({
                    "Prior": name,
                    "Hidden": n_hid,
                    "Sigma": sig,
                    "Top-1": acc1,
                    "Top-2": acc2
                })

# Find best config
best = max(results_tuning, key=lambda x: x['Top-2'])
print(f"\nBest Overall Config: {best}")

--- Starting Full Grid Search ---
Prior: Uniform | Hidden: 5 | Sigma: 0.1 | Top-1: 28.83% | Top-2: 42.94%
Prior: Uniform | Hidden: 5 | Sigma: 0.5 | Top-1: 34.36% | Top-2: 54.60%
Prior: Uniform | Hidden: 10 | Sigma: 0.1 | Top-1: 28.83% | Top-2: 42.94%
Prior: Uniform | Hidden: 10 | Sigma: 0.5 | Top-1: 34.97% | Top-2: 55.21%
Prior: Dataset | Hidden: 5 | Sigma: 0.1 | Top-1: 28.83% | Top-2: 42.94%
Prior: Dataset | Hidden: 5 | Sigma: 0.5 | Top-1: 33.74% | Top-2: 54.60%
Prior: Dataset | Hidden: 10 | Sigma: 0.1 | Top-1: 28.83% | Top-2: 42.94%
Prior: Dataset | Hidden: 10 | Sigma: 0.5 | Top-1: 33.74% | Top-2: 55.21%
Prior: Real-World | Hidden: 5 | Sigma: 0.1 | Top-1: 28.83% | Top-2: 42.94%
Prior: Real-World | Hidden: 5 | Sigma: 0.5 | Top-1: 33.74% | Top-2: 53.99%
Prior: Real-World | Hidden: 10 | Sigma: 0.1 | Top-1: 28.83% | Top-2: 42.94%
Prior: Real-World | Hidden: 10 | Sigma: 0.5 | Top-1: 34.97% | Top-2: 52.76%

Best Overall Config: {'Prior': 'Uniform', 'Hidden': 10, 'Sigma': 0.5, 'Top-1': np.f

Save model

In [8]:
import joblib
import json

best_prior   = best["Prior"]
best_hidden  = best["Hidden"]
best_sigma   = best["Sigma"]

print("\nRefitting best BNN model for saving...")
print(f" Best Prior: {best_prior}, Hidden: {best_hidden}, Sigma: {best_sigma}")

# Retrieve the mean vector for the chosen prior
prior_mu = prior_experiments[best_prior]

coords = {
    "classes": classes,
    "features": [f"f{i}" for i in range(X_train_scaled.shape[1])],
    "hidden":   [f"h{i}" for i in range(best_hidden)]
}

with pm.Model(coords=coords) as final_model:
    X_data = pm.Data("X_data", X_train_scaled)
    y_data = pm.Data("y_data", y_train_int)

    w1 = pm.Normal("w1", 0, best_sigma, dims=("features", "hidden"))
    b1 = pm.Normal("b1", 0, best_sigma, dims="hidden")
    act1 = pm_math.tanh(pm_math.dot(X_data, w1) + b1)

    w2 = pm.Normal("w2", 0, best_sigma, dims=("hidden", "classes"))
    b2 = pm.Normal("b2", mu=prior_mu, sigma=1.0, dims="classes")

    logits = pm_math.dot(act1, w2) + b2
    probs  = pm.Deterministic("probs", pm_math.softmax(logits, axis=1))
    pm.Categorical("y_obs", logit_p=logits, observed=y_data)

    approx = pm.fit(n=30000, method='advi', progressbar=False, random_seed=123)
    trace  = approx.sample(draws=1000)

# Extract POSTERIOR MEANS (for test-time prediction)
w1_mean = trace.posterior["w1"].mean(dim=["chain","draw"]).values
b1_mean = trace.posterior["b1"].mean(dim=["chain","draw"]).values
w2_mean = trace.posterior["w2"].mean(dim=["chain","draw"]).values
b2_mean = trace.posterior["b2"].mean(dim=["chain","draw"]).values

# Save neural network parameters
np.save("w1_mean.npy", w1_mean)
np.save("b1_mean.npy", b1_mean)
np.save("w2_mean.npy", w2_mean)
np.save("b2_mean.npy", b2_mean)

# Save classes
np.save("classes.npy", np.array(classes))

# Save feature names in order
feature_names = [f"f{i}" for i in range(X_train_scaled.shape[1])]
with open("feature_names.json", "w") as f:
    json.dump(feature_names, f)

# Save hidden layer size
with open("hidden_size.json", "w") as f:
    json.dump({"hidden": best_hidden}, f)

# Save the scaler
joblib.dump(scaler, "scaler.pkl")

print("\nSaved BNN model parameters and metadata:")
print("  w1_mean.npy, b1_mean.npy, w2_mean.npy, b2_mean.npy")
print("  classes.npy, feature_names.json, hidden_size.json, scaler.pkl")


Refitting best BNN model for saving...
 Best Prior: Uniform, Hidden: 10, Sigma: 0.5

Saved BNN model parameters and metadata:
  w1_mean.npy, b1_mean.npy, w2_mean.npy, b2_mean.npy
  classes.npy, feature_names.json, hidden_size.json, scaler.pkl


# Weighted Bayesian Neural Network (Complex Model Comparison)

In [ ]:
import pytensor.tensor as pt
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

# Setup Class Weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_int), y=y_train_int)
weights_tensor = pt.as_tensor_variable(class_weights)

results_bnn = []
print("--- Starting Weighted BNN Prior Comparison ---")

# Heuristic parameters (tuned for stability)
n_hidden = 40
weight_sigma = 0.5

for name, prior_mu in prior_experiments.items():
    print(f"Running: Weighted BNN + {name} Prior...")

    coords = {
        "classes": classes,
        "features": [f"f{i}" for i in range(X_train_scaled.shape[1])],
        "hidden": [f"h{i}" for i in range(n_hidden)]
    }

    with pm.Model(coords=coords) as bnn_model:
        X_data = pm.Data("X_data", X_train_scaled)
        y_data = pm.Data("y_data", y_train_int)

        # Network Layers
        w1 = pm.Normal("w1", 0, weight_sigma, dims=("features", "hidden"))
        b1 = pm.Normal("b1", 0, weight_sigma, dims="hidden")
        act1 = pm_math.tanh(pm_math.dot(X_data, w1) + b1)

        w2 = pm.Normal("w2", 0, weight_sigma, dims=("hidden", "classes"))

        # Inject Specific Prior into Bias
        b2 = pm.Normal("b2", mu=prior_mu, sigma=1.0, dims="classes")

        logits = pm_math.dot(act1, w2) + b2

        # Probabilities & Weighted Likelihood
        probs = pm.Deterministic("probs", pm.math.softmax(logits, axis=1))

        # Manual Log-Likelihood for weighting
        true_probs = probs[pt.arange(y_data.shape[0]), y_data]
        weighted_ll = pm.math.log(true_probs + 1e-8) * weights_tensor[y_data]
        pm.Potential("weighted_obs", weighted_ll)

        # Inference
        approx = pm.fit(n=40000, method='advi', progressbar=False, random_seed=42)
        trace = approx.sample(draws=500)

        # Evaluation
        pm.set_data({"X_data": X_val_scaled, "y_data": y_val_int})
        post_pred = pm.sample_posterior_predictive(trace, var_names=["probs"], progressbar=False)

        # Metrics
        mean_probs = post_pred.posterior_predictive["probs"].mean(dim=["chain", "draw"]).values

        acc1 = top_k_accuracy_score(y_val_int, mean_probs, k=1)
        acc2 = top_k_accuracy_score(y_val_int, mean_probs, k=2)

        print(f"Result: Top-1={acc1:.2%} | Top-2={acc2:.2%}")
        results_bnn.append({"Prior": name, "Top-1": acc1, "Top-2": acc2})

print("\n" + "="*30)
print(pd.DataFrame(results_bnn).to_string(index=False))

--- Starting Weighted BNN Prior Comparison ---
Running: Weighted BNN + Uniform Prior...


/tmp/ipython-input-2847593379.py:55: UserWarning: The effect of Potentials on other parameters is ignored during posterior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  post_pred = pm.sample_posterior_predictive(trace, var_names=["probs"], progressbar=False)


Result: Top-1=22.09% | Top-2=38.65%
Running: Weighted BNN + Dataset Prior...


/tmp/ipython-input-2847593379.py:55: UserWarning: The effect of Potentials on other parameters is ignored during posterior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  post_pred = pm.sample_posterior_predictive(trace, var_names=["probs"], progressbar=False)


Result: Top-1=21.47% | Top-2=38.04%
Running: Weighted BNN + Real-World Prior...


/tmp/ipython-input-2847593379.py:55: UserWarning: The effect of Potentials on other parameters is ignored during posterior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  post_pred = pm.sample_posterior_predictive(trace, var_names=["probs"], progressbar=False)


Result: Top-1=20.86% | Top-2=38.04%

     Prior    Top-1    Top-2
   Uniform 0.220859 0.386503
   Dataset 0.214724 0.380368
Real-World 0.208589 0.380368
